# Polaris Purge Practice — Catalog / Table / View

**Purpose:** Practice drop-with-purge on local OrbStack cluster  
**Observe:** MinIO before/after each purge via http://192.168.139.2:9001  

## What we do
```
Setup:    cleanup any leftover resources → create fresh catalog/ns/table/view
Part 1b:  write data to table (pyarrow + Iceberg REST commit)
Part 2:   observe MinIO before purge
Part 3:   view purge  → no MinIO change
Part 4:   table purge → Parquet files removed
Part 5:   catalog purge → base location removed
Part 6:   verify everything gone
```


In [1]:
import sys
sys.path.insert(0, '.')
from polaris_test_utils import *

tok = root_token()
print(f"✅ Root token acquired")
print(f"Polaris: {POLARIS_URL}")
print(f"MinIO:   http://192.168.139.2:9001")

# Practice resource names
PRACTICE_CATALOG  = "purge-practice"
PRACTICE_NS       = "practice-ns"
PRACTICE_TABLE    = "practice-table"
PRACTICE_VIEW     = "practice-view"
PRACTICE_LOCATION = "s3a://data-catalog-bucket/purge-practice/"

# MinIO credentials (direct — STS not used)
MINIO_ENDPOINT   = "http://192.168.139.2:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "bhEenDzTEAVPKbukhuY0tRY4WjpZXzh0"


✅ Polaris test utils loaded
   Polaris: http://192.168.139.2:8181  | Realm: POLARIS
   OpenSearch: https://localhost:9200
✅ Root token acquired
Polaris: http://192.168.139.2:8181
MinIO:   http://192.168.139.2:9001


## Cleanup — remove any leftover resources from previous runs

In [4]:
# Cleanup: delete practice catalog (and everything inside it) if exists
print("Cleaning up leftover resources...")

catalogs = requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok)).json().get("catalogs", [])
if any(c["name"] == PRACTICE_CATALOG for c in catalogs):
    # Delete tables + views + namespaces first
    ns_r = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces", headers=h(tok))
    for ns in ns_r.json().get("namespaces", []):
        ns_name = ns[0] if isinstance(ns, list) else ns

        # Tables
        t_r = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}/tables", headers=h(tok))
        for t in t_r.json().get("identifiers", []):
            requests.delete(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}/tables/{t['name']}", headers=h(tok))
            print(f"  deleted table: {t['name']}")

        # Views
        v_r = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}/views", headers=h(tok))
        for v in v_r.json().get("identifiers", []):
            requests.delete(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}/views/{v['name']}", headers=h(tok))
            print(f"  deleted view: {v['name']}")

        # Namespace
        requests.delete(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{ns_name}", headers=h(tok))
        print(f"  deleted namespace: {ns_name}")

    # Catalog
    requests.delete(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}", headers=h(tok))
    print(f"  deleted catalog: {PRACTICE_CATALOG}")
else:
    print("  nothing to clean up")

print("✅ Ready for fresh run")


Cleaning up leftover resources...
  deleted view: practice-view
  deleted namespace: practice-ns
  deleted catalog: purge-practice
✅ Ready for fresh run


## Part 1 — Create catalog, namespace, table, view

In [3]:
# Step 1: Create practice catalog (purge DISABLED initially)
print("Step 1: Create practice catalog")
r = requests.post(f"{BASE_MGMT}/catalogs", headers=h(tok), json={
    "catalog": {
        "name": PRACTICE_CATALOG,
        "type": "INTERNAL",
        "properties": {
            "default-base-location": PRACTICE_LOCATION,
            "polaris.config.drop-with-purge.enabled": "false",
        },
        "storageConfigInfo": {
            "storageType": "S3",
            "allowedLocations": [PRACTICE_LOCATION],
            "pathStyleAccess": True,
            "endpoint": "http://192.168.139.2:9000",
            "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000",
        },
    }
})
print_response(r, "Create catalog")
catalog_version = r.json().get("catalog", {}).get("entityVersion", 1)
print(f"\n✅ entityVersion: {catalog_version}")


Step 1: Create practice catalog

Create catalog:
  Status:     409
  Request-Id: 626665e0-9248-4452-921e-0535b85a9a56_0000000000000000262
  Body: {
  "error": {
    "message": "Cannot create Catalog purge-practice. Catalog already exists or resolution failed",
    "type": "AlreadyExistsException",
    "code": 409
  }
}

✅ entityVersion: 1


In [ ]:
# Step 2: Create namespace
print("Step 2: Create namespace")
r = create_namespace(catalog=PRACTICE_CATALOG, ns=PRACTICE_NS, token=tok)
print_response(r, "Create namespace")


In [ ]:
# Step 3: Create table
print("Step 3: Create table")
r = requests.post(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables",
    headers=h(tok),
    json={
        "name": PRACTICE_TABLE,
        "location": f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_TABLE}/",
        "schema": {
            "type": "struct",
            "fields": [
                {"id": 1, "name": "id",     "type": "long",   "required": True},
                {"id": 2, "name": "name",   "type": "string", "required": False},
                {"id": 3, "name": "amount", "type": "double", "required": False},
            ]
        },
        "properties": {"write.format.default": "parquet"}
    }
)
print_response(r, "Create table")
# Save current snapshot for commit requirement
table_meta      = r.json().get("metadata", {})
current_snapshot = table_meta.get("current-snapshot-id", -1)
table_location   = table_meta.get("location", f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_TABLE}")
print(f"\n✅ Table created  current-snapshot: {current_snapshot}")


In [ ]:
# Step 4: Create view
print("Step 4: Create view")
r = requests.post(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/views",
    headers=h(tok),
    json={
        "name": PRACTICE_VIEW,
        "default-namespace": [PRACTICE_NS],
        "schema": {
            "type": "struct",
            "schema-id": 0,
            "fields": [
                {"id": 1, "name": "id",   "type": "long",   "required": True},
                {"id": 2, "name": "name", "type": "string", "required": False},
            ]
        },
        "view-version": {
            "version-id": 1,
            "timestamp-ms": int(__import__("time").time() * 1000),
            "schema-id": 0,
            "default-namespace": [PRACTICE_NS],
            "summary": {"engine": "spark"},
            "representations": [{
                "type": "sql",
                "sql": f"SELECT id, name FROM {PRACTICE_TABLE}",
                "dialect": "spark"
            }]
        },
        "location": f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_VIEW}/",
        "properties": {}
    }
)
print_response(r, "Create view")


## Part 1b — Write data to table via Iceberg REST

```
Iceberg v2 write flow (without Spark):
  Step A: write Parquet file to MinIO (pyarrow + s3fs)
  Step B: write manifest + manifest-list Avro files to MinIO
  Step C: commit snapshot via updateTable API

Polaris 1.3.0 commit rules (confirmed from testing):
  → requirements can be []
  → add-snapshot only (no set-snapshot-ref in same request)
  → no sequence-number field (Polaris manages internally)
  → manifest-list must be real path (not empty string)
```


In [ ]:
# Step A: Write Parquet file to MinIO
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs, uuid, io, time
try:
    import fastavro
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "fastavro", "--quiet"])
    import fastavro

print("Step A: Write Parquet file to MinIO")

table_data = pa.table({
    "id":     pa.array([1, 2, 3, 4, 5], type=pa.int64()),
    "name":   pa.array(["alice", "bob", "carol", "dave", "eve"], type=pa.string()),
    "amount": pa.array([100.0, 200.5, 300.0, 150.75, 250.0], type=pa.float64()),
})

file_uuid    = uuid.uuid4().hex
data_dir     = f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_TABLE}/data"
parquet_path = f"{data_dir}/{file_uuid}.parquet"
s3_path      = parquet_path.replace("s3a://", "s3://")

fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    use_ssl=False,
    client_kwargs={"endpoint_url": MINIO_ENDPOINT}
)

with fs.open(s3_path, "wb") as f_out:
    pq.write_table(table_data, f_out)

FILE_SIZE_BYTES = fs.info(s3_path)["size"]
print(f"  ✅ Parquet written: {FILE_SIZE_BYTES} bytes, {len(table_data)} rows")
print(f"  path: {parquet_path}")
print()
print("📸 MinIO: data file visible now (no snapshot yet)")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")


In [ ]:
# Step B: Write Iceberg v2 manifest + manifest-list to MinIO
print("Step B: Write manifest + manifest-list")

snapshot_id = int(time.time() * 1000)
commit_ts   = snapshot_id
seq_num     = 1
meta_dir    = f"{PRACTICE_LOCATION}{PRACTICE_NS}/{PRACTICE_TABLE}/metadata"

MANIFEST_SCHEMA = {
    "type": "record", "name": "manifest_entry",
    "fields": [
        {"name": "status",          "type": "int"},
        {"name": "snapshot_id",     "type": ["null","long"], "default": None},
        {"name": "sequence_number", "type": ["null","long"], "default": None},
        {"name": "file_sequence_number","type":["null","long"],"default":None},
        {"name": "data_file", "type": {
            "type": "record", "name": "r2",
            "fields": [
                {"name": "content",            "type": "int"},
                {"name": "file_path",          "type": "string"},
                {"name": "file_format",        "type": "string"},
                {"name": "partition",          "type": {"type":"record","name":"r102","fields":[]}},
                {"name": "record_count",       "type": "long"},
                {"name": "file_size_in_bytes", "type": "long"},
                {"name": "column_sizes",       "type":["null",{"type":"array","items":{"type":"record","name":"k81v81","fields":[{"name":"key","type":"int"},{"name":"value","type":"long"}]}}],"default":None},
                {"name": "value_counts",       "type":["null",{"type":"array","items":"k81v81"}],"default":None},
                {"name": "null_value_counts",  "type":["null",{"type":"array","items":"k81v81"}],"default":None},
                {"name": "nan_value_counts",   "type":["null",{"type":"array","items":"k81v81"}],"default":None},
                {"name": "lower_bounds",       "type":["null",{"type":"array","items":{"type":"record","name":"k81v100","fields":[{"name":"key","type":"int"},{"name":"value","type":"bytes"}]}}],"default":None},
                {"name": "upper_bounds",       "type":["null",{"type":"array","items":"k81v100"}],"default":None},
                {"name": "key_metadata",       "type":["null","bytes"],"default":None},
                {"name": "split_offsets",      "type":["null",{"type":"array","items":"long"}],"default":None},
                {"name": "equality_ids",       "type":["null",{"type":"array","items":"int"}],"default":None},
                {"name": "sort_order_id",      "type":["null","int"],"default":None},
            ]
        }}
    ]
}

manifest_entry = {
    "status": 1, "snapshot_id": snapshot_id,
    "sequence_number": seq_num, "file_sequence_number": seq_num,
    "data_file": {
        "content": 0, "file_path": parquet_path, "file_format": "PARQUET",
        "partition": {}, "record_count": 5, "file_size_in_bytes": FILE_SIZE_BYTES,
        "column_sizes": None, "value_counts": None, "null_value_counts": None,
        "nan_value_counts": None, "lower_bounds": None, "upper_bounds": None,
        "key_metadata": None, "split_offsets": None, "equality_ids": None, "sort_order_id": 0,
    }
}

manifest_path   = f"{meta_dir}/{uuid.uuid4().hex}-m0.avro"
s3_manifest     = manifest_path.replace("s3a://", "s3://")
buf = io.BytesIO()
fastavro.writer(buf, fastavro.parse_schema(MANIFEST_SCHEMA), [manifest_entry],
    metadata={
        "schema": '{"type":"struct","schema-id":0,"fields":[{"id":1,"name":"id","required":true,"type":"long"},{"id":2,"name":"name","required":false,"type":"string"},{"id":3,"name":"amount","required":false,"type":"double"}]}',
        "partition-spec": '[]', "partition-spec-id": '0', "format-version": '2',
        "content": 'data', "sequence-number": str(seq_num),
        "min-sequence-number": str(seq_num), "snapshot-id": str(snapshot_id),
        "added-data-files": '1', "existing-data-files": '0', "deleted-data-files": '0',
        "added-rows": '5', "existing-rows": '0', "deleted-rows": '0',
    })
manifest_bytes = buf.getvalue()
manifest_size  = len(manifest_bytes)
with fs.open(s3_manifest, "wb") as f: f.write(manifest_bytes)
print(f"  ✅ manifest: {s3_manifest} ({manifest_size}B)")

MANIFEST_LIST_SCHEMA = {
    "type": "record", "name": "manifest_file",
    "fields": [
        {"name": "manifest_path",              "type": "string"},
        {"name": "manifest_length",             "type": "long"},
        {"name": "partition_spec_id",           "type": "int"},
        {"name": "content",                     "type": "int",  "default": 0},
        {"name": "sequence_number",             "type": "long", "default": 0},
        {"name": "min_sequence_number",         "type": "long", "default": 0},
        {"name": "added_snapshot_id",           "type": "long"},
        {"name": "added_data_files_count",      "type": "int"},
        {"name": "existing_data_files_count",   "type": "int"},
        {"name": "deleted_data_files_count",    "type": "int"},
        {"name": "added_rows_count",            "type": "long"},
        {"name": "existing_rows_count",         "type": "long"},
        {"name": "deleted_rows_count",          "type": "long"},
        {"name": "partitions",  "type":["null",{"type":"array","items":{"type":"record","name":"r508","fields":[{"name":"contains_null","type":"boolean"},{"name":"contains_nan","type":["null","boolean"],"default":None},{"name":"lower_bound","type":["null","bytes"],"default":None},{"name":"upper_bound","type":["null","bytes"],"default":None}]}}],"default":None},
        {"name": "key_metadata","type":["null","bytes"],"default":None},
    ]
}

manifest_list_path = f"{meta_dir}/snap-{snapshot_id}-{uuid.uuid4().hex}.avro"
s3_ml = manifest_list_path.replace("s3a://", "s3://")
buf2 = io.BytesIO()
fastavro.writer(buf2, fastavro.parse_schema(MANIFEST_LIST_SCHEMA), [{
    "manifest_path": manifest_path, "manifest_length": manifest_size,
    "partition_spec_id": 0, "content": 0, "sequence_number": seq_num,
    "min_sequence_number": seq_num, "added_snapshot_id": snapshot_id,
    "added_data_files_count": 1, "existing_data_files_count": 0, "deleted_data_files_count": 0,
    "added_rows_count": 5, "existing_rows_count": 0, "deleted_rows_count": 0,
    "partitions": None, "key_metadata": None,
}])
with fs.open(s3_ml, "wb") as f: f.write(buf2.getvalue())
print(f"  ✅ manifest-list: {s3_ml}")
print()
print("📸 MinIO: metadata/ should now have .avro files")


In [ ]:
# Step C: Commit snapshot (Polaris 1.3.0 confirmed working body)
print("Step C: Commit snapshot")

r_commit = requests.post(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}",
    headers=h(tok),
    json={
        "identifier": {"namespace": [PRACTICE_NS], "name": PRACTICE_TABLE},
        "requirements": [],
        "updates": [{
            "action": "add-snapshot",
            "snapshot": {
                "snapshot-id":   snapshot_id,
                "timestamp-ms":  commit_ts,
                "summary": {
                    "operation":         "append",
                    "added-data-files":  "1",
                    "added-records":     "5",
                    "added-files-size":  str(FILE_SIZE_BYTES),
                    "total-data-files":  "1",
                    "total-records":     "5",
                },
                "schema-id":     0,
                "manifest-list": manifest_list_path,
            }
        }]
    }
)
print_response(r_commit, "Commit snapshot")

if r_commit.status_code in (200, 201, 204):
    # Set main branch ref (separate request)
    r_ref = requests.post(
        f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}",
        headers=h(tok),
        json={
            "identifier": {"namespace": [PRACTICE_NS], "name": PRACTICE_TABLE},
            "requirements": [],
            "updates": [{
                "action": "set-snapshot-ref",
                "ref-name": "main",
                "type": "branch",
                "snapshot-id": snapshot_id,
            }]
        }
    )
    current = r_ref.json().get("metadata", {}).get("current-snapshot-id", -1)
    print(f"\n  set-snapshot-ref: HTTP {r_ref.status_code}")
    print(f"  current-snapshot: {current}")
    print(f"\n✅ 5 rows committed — visible to Spark/Trino via Polaris")


In [ ]:
# Step D: Verify snapshot
print("Step D: Verify")
r = requests.get(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}",
    headers=h(tok)
)
meta      = r.json().get("metadata", {})
snapshots = meta.get("snapshots", [])
current   = meta.get("current-snapshot-id", -1)
refs      = meta.get("refs", {})

print(f"  snapshots:        {len(snapshots)}")
print(f"  current-snapshot: {current}")
print(f"  refs:             {refs}")
for s in snapshots:
    sm = s.get("summary", {})
    print(f"  → id={s.get('snapshot-id')} op={sm.get('operation')} records={sm.get('added-records','?')}")

print()
if current == snapshot_id:
    print("✅ Snapshot confirmed — table has 5 rows")
    print()
    print("📸 MinIO final state before purge:")
    print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
    print("   data/     → .parquet (5 rows)")
    print("   metadata/ → -m0.avro (manifest)")
    print("              snap-*.avro (manifest-list)")
    print("              .metadata.json (table metadata)")
else:
    print(f"⚠️  current-snapshot={current} ≠ {snapshot_id}")


## Part 2 — Observe MinIO before purge

In [ ]:
# Verify all entities exist
print("Verify entities exist before purge:")
r_t = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}", headers=h(tok))
r_v = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/views/{PRACTICE_VIEW}", headers=h(tok))
print(f"  Table {PRACTICE_TABLE}: HTTP {r_t.status_code} {'✅' if r_t.status_code == 200 else '❌'}")
print(f"  View  {PRACTICE_VIEW}:  HTTP {r_v.status_code} {'✅' if r_v.status_code == 200 else '❌'}")
print()
print("📸 Check MinIO NOW (before any purge)")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")


## Part 3 — View purge (no MinIO effect)

In [ ]:
# Enable DROP_WITH_PURGE on catalog
print("Enable DROP_WITH_PURGE on catalog")
r_get = requests.get(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}", headers=h(tok))
version = r_get.json().get("catalog", {}).get("entityVersion", 1)
r_patch = requests.patch(
    f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}",
    headers=h(tok),
    json={
        "currentEntityVersion": version,
        "properties": {"polaris.config.drop-with-purge.enabled": "true"}
    }
)
print_response(r_patch, "Enable purge")


In [ ]:
# Drop VIEW with purge
print("Drop view WITH purgeRequested=true")
r = requests.delete(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/views/{PRACTICE_VIEW}?purgeRequested=true",
    headers=h(tok)
)
print_response(r, "Drop view (purge)")
print()
print("📸 Check MinIO AFTER view purge")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print("Expected: NO change (views have no files)")


## Part 4 — Table purge (Parquet files removed from MinIO)

In [ ]:
# Drop TABLE with purge
print("Drop table WITH purgeRequested=true")
print()
print("📸 MinIO BEFORE table purge — note files present")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print()

r = requests.delete(
    f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}?purgeRequested=true",
    headers=h(tok)
)
print_response(r, "Drop table (purge)")
print()
print("📸 MinIO AFTER table purge")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print("Expected: data/ + metadata/ GONE")


In [ ]:
# Verify table gone
r = requests.get(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}/tables/{PRACTICE_TABLE}", headers=h(tok))
print(f"Table after purge: HTTP {r.status_code} {'✅ 404=gone' if r.status_code == 404 else '❌ still exists'}")


## Part 5 — Catalog purge (base location removed from MinIO)

In [ ]:
# Drop namespace first (required before catalog)
print("Drop namespace")
r = requests.delete(f"{BASE_CAT}/{PRACTICE_CATALOG}/namespaces/{PRACTICE_NS}", headers=h(tok))
print_response(r, "Drop namespace")


In [ ]:
# Drop CATALOG with purge
print("Drop catalog WITH purgeRequested=true")
print()
print("📸 MinIO BEFORE catalog purge")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print()

r = requests.delete(
    f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}?purgeRequested=true",
    headers=h(tok)
)
print_response(r, "Drop catalog (purge)")
print()
print("📸 MinIO AFTER catalog purge")
print(f"   http://192.168.139.2:9001/browser/data-catalog-bucket/purge-practice%2F")
print("Expected: purge-practice/ prefix GONE")


## Part 6 — Verification

In [ ]:
print("Final verification")
r = requests.get(f"{BASE_MGMT}/catalogs/{PRACTICE_CATALOG}", headers=h(tok))
print(f"Catalog: HTTP {r.status_code} {'✅ 404=gone' if r.status_code == 404 else '❌ still exists'}")

all_cats = [c["name"] for c in requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok)).json().get("catalogs", [])]
print(f"All catalogs: {all_cats}")
print(f"purge-practice present: {'❌' if PRACTICE_CATALOG in all_cats else '✅ removed'}")


## Summary — What we learned

```
View  + purge → PostgreSQL metadata removed, MinIO unchanged ✅
Table + purge → PostgreSQL metadata + MinIO files removed ✅
Catalog+purge → PostgreSQL metadata + MinIO base location removed ✅

Order (always bottom-up):
  table → namespace → catalog

Polaris 1.3.0 commit quirks:
  → no sequence-number in snapshot body
  → add-snapshot and set-snapshot-ref must be separate requests
  → manifest-list must be a real written path

Production rule:
  → disable DROP_WITH_PURGE on bronze/silver/gold
  → enable only on test/watchdog catalogs
```
